# Google AI Pro × Colab A100 × Strata：跑跑看 125B MoE

**2026 年 10 月 10 日實測環境：** Colab A100-SXM4 40GB、系統 RAM 約 83 GiB、CUDA 13（驅動 580.82.07）、[Strata](https://github.com/Niko1221/Strata)、**Qwen3.8-Flash-Next（125B MoE）IQ2_XS（約 2-bit 量化）**、32K context、關閉圖片輸入。

Google AI Pro 方案內含 Colab 運算單元，但不保證分配到 A100。請先在 Colab 選 **［執行階段］→［變更執行階段類型］→［A100 GPU］**，若可選 **高 RAM** 請開啟。RAM、磁碟空間不及測試環境時可能無法完成安裝。

**所需空間與時間：** 首次需下載約 68GB 模型、產生中間資料與暫存檔，必要時還要編譯 CUDA 引擎，會花不少時間。磁碟請預留 120GB 以上（實際需求依 Strata 版本而異）。使用高階 GPU 期間會持續消耗運算單元。Colab 不是常駐伺服器，請只用於**在 Colab 內做個人的功能與速度驗證**。

**實測例（單次、不同條件）：** 長文 6,886 輸入 token 時 Prefill **2,572 tok/s**；最多輸出 64 token 時 Decode **116.9 tok/s**。另一次 1,024 token 輸出測試 Decode **68.0 tok/s**。**116.9 tok/s 不是連續生成 1024 token 的速度。**

參考：[Strata 官方](https://github.com/Niko1221/Strata) / [Google 的 Colab 方案公告](https://developers.googleblog.com/colab-is-now-part-of-your-google-ai-plan/) / [Colab FAQ](https://research.google.com/colaboratory/faq.html?hl=zh-TW)

## 1. 檢查 GPU、RAM、SSD 並取得 Strata

優先使用 `8080`。若已被其他服務佔用，自動改用 `8081` 或 `8082`。成功環境的關鍵是用 `STRATA_UV=1` 建立 Python 環境。

In [ ]:
import os, sys, socket, subprocess
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
driver = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], text=True).strip().splitlines()[0]
assert "A100" in gpu, f"本筆記本的驗證目標是 A100。實際分配：{gpu}"
assert int(driver.split(".")[0]) >= 580, f"CUDA 13 需要 NVIDIA 驅動 580 以上。目前：{driver}"

def port_is_free(p):
    with socket.socket() as sock:
        try:
            sock.bind(("127.0.0.1", p))
            return True
        except OSError:
            return False

PORT = next((p for p in (8080, 8081, 8082) if port_is_free(p)), None)
assert PORT is not None, "8080～8082 埠全部被佔用"
os.environ["STRATA_PORT"] = str(PORT)
print("Strata 使用的埠：", PORT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
root = Path("/content/Strata")
if not (root / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Niko1221/Strata.git", str(root)], check=True)

print("取得完成。請在下一格執行安裝。")

## 2. 安裝 Strata 與量化模型

配合這次的 A100 40GB／83GB RAM，使用 **IQ2_XS（約 2-bit）**、**32,768 context**、CUDA 13。為避開 Colab 預設 Python 與 `venv` 的相容問題，設定 `STRATA_UV=1`。

**首次會花很久。** 包括約 68GB 模型下載、視需要編譯 CUDA、準備模型。下面這格不用原本的 `subprocess.run()`，改用 Colab 的 `!` 指令執行以顯示進度 log。`--no-start` 表示安裝完不啟動伺服器。中途中斷的話，不要刪除執行階段，重跑同一格通常能續傳。

In [ ]:
import os
from pathlib import Path

os.environ["STRATA_UV"] = "1"
os.environ["PATH"] = (
    "/usr/local/bin:" + os.path.expanduser("~/.local/bin")
    + ":" + os.environ.get("PATH", "")
)
print(f"開始安裝 Strata（埠 {PORT}）。首次會花較長時間。", flush=True)

# 用 Colab 的 ! 指令執行，下載與編譯的 log 比較好閱讀。
!cd /content/Strata && bash setup.sh \
  --setup --yes \
  --backend cuda --cuda 13 --gpu 0 \
  --family qwen --model IQ2_XS \
  --context 32768 --vision no \
  --data-dir /content/Strata-data \
  --host 127.0.0.1 --port {PORT} \
  --no-browser --no-start

assert Path("/content/Strata/run-iq2_xs.sh").is_file(), (
    "安裝尚未完成，請檢查上方 log 末段。"
)
print("已確認啟動腳本，請在下一格啟動伺服器。")

## 3. 啟動伺服器並等待模型載入

首次啟動會把數十 GB 權重配置到 RAM／VRAM，**可能要 2 分鐘以上**。請等到顯示 `loaded: True`。

In [ ]:
import requests, time, subprocess
from pathlib import Path

BASE = f"http://127.0.0.1:{PORT}"
s = requests.Session()
s.trust_env = False

def get_health():
    try:
        r = s.get(BASE + "/health", timeout=3)
        if r.ok:
            j = r.json()
            if j.get("service") == "strata":
                return j
    except (requests.RequestException, ValueError):
        pass
    return None

health = get_health()
if not (health and health.get("loaded")):
    script = Path("/content/Strata/run-iq2_xs.sh")
    assert script.exists(), "安裝未完成（找不到 run-iq2_xs.sh）"
    log_path = Path(f"/content/strata-{PORT}.log")
    with log_path.open("a") as log:
        proc = subprocess.Popen(["bash", str(script)], cwd="/content/Strata",
                                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print(f"伺服器已啟動 PID={proc.pid}，載入中…")
    for i in range(120):  # 最多 10 分鐘
        health = get_health()
        if health and health.get("loaded"):
            break
        if proc.poll() is not None:
            print(log_path.read_text(errors="replace")[-5000:])
            raise RuntimeError(f"Strata 已結束（exit {proc.returncode}）")
        if i % 6 == 0:
            print(f"已過 {i * 5} 秒，等待模型載入…")
        time.sleep(5)
    else:
        print(log_path.read_text(errors="replace")[-5000:])
        raise TimeoutError("等了 10 分鐘 Strata 仍未啟動，請檢查 log。")

print("READY:", health)
print("OpenAI 相容 API:", BASE + "/v1")

## 4. 用 trycloudflare 對外公開 API

用 Cloudflare Quick Tunnel 取得臨時的 `https://xxxx.trycloudflare.com` 網址，讓 Colab 外部（自己電腦、手機、其他工具）也能呼叫 OpenAI 相容 API。不需要 Cloudflare 帳號。

注意：
- **網址公開、無認證。** 拿到網址的人都能用你的 GPU。不要外流，用完就關。
- 每次啟動網址都會變；執行階段中斷後即失效。
- Quick Tunnel 定位為測試用，有並行請求上限，官方文件註明不支援 SSE。`stream: true` 經 tunnel 可能無法逐字輸出，必要時改用 `stream: false`。
- Colab FAQ 對部分遠端存取用途有限制，只做短時間個人測試。

In [ ]:
import re, time, subprocess, requests
from pathlib import Path

CF = Path("/usr/local/bin/cloudflared")
if not CF.exists():
    subprocess.run(["wget", "-q", "-O", str(CF),
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    CF.chmod(0o755)

# 重跑本格時先關掉舊的 tunnel
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)

cf_log = Path("/content/cloudflared.log")
cf_proc = subprocess.Popen(
    [str(CF), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=cf_log.open("w"), stderr=subprocess.STDOUT, start_new_session=True,
)

PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text(errors="replace"))
    if m:
        PUBLIC_URL = m.group(0)
        break
    if cf_proc.poll() is not None:
        print(cf_log.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"cloudflared 已結束（exit {cf_proc.returncode}）")
    time.sleep(1)
assert PUBLIC_URL, "60 秒內沒拿到 trycloudflare 網址，請檢查 /content/cloudflared.log"

# DNS 生效需要幾秒，等 /health 通了才算完成
for _ in range(30):
    try:
        r = requests.get(PUBLIC_URL + "/health", timeout=5)
        if r.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("警告：外部 /health 尚未回應，可能仍在 DNS 生效中，稍後再試。")

print("公開網址:", PUBLIC_URL)
print("OpenAI 相容 API:", PUBLIC_URL + "/v1")
print("模型名稱: qwen3.8-flash-next-iq2_xs")

### 從 Colab 外部呼叫

把 `<PUBLIC_URL>` 換成上一格印出的網址。

```bash
curl <PUBLIC_URL>/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen3.8-flash-next-iq2_xs","messages":[{"role":"user","content":"你好"}],"max_tokens":128,"stream":false}'
```

```python
from openai import OpenAI
client = OpenAI(base_url="<PUBLIC_URL>/v1", api_key="dummy")
r = client.chat.completions.create(
    model="qwen3.8-flash-next-iq2_xs",
    messages=[{"role": "user", "content": "你好"}],
    max_tokens=128,
)
print(r.choices[0].message.content)
```

其他支援 OpenAI 相容端點的工具（Open WebUI、Cherry Studio 等），Base URL 填 `<PUBLIC_URL>/v1`，API key 隨便填。

## 5. 串流生成文字並顯示速度

最後會顯示引擎回傳的 Prefill、Decode 內部量測值。首次執行與短文的速度會有波動，必要時多跑幾次。把 `TARGET` 改成 `PUBLIC_URL` 可經由 tunnel 測試。

In [ ]:
import requests, time, json

TARGET = BASE  # 改成 PUBLIC_URL 可測試 tunnel

payload = {
    "model": "qwen3.8-flash-next-iq2_xs",
    "messages": [{"role": "user", "content":
                  "請用繁體中文，舉具體例子說明 AI 會如何改變化學廠商的研發。"}],
    "max_tokens": 512,
    "reasoning_effort": "none",
    "temperature": 0,
    "stream": True,
}

start = time.perf_counter()
first_text = None
usage, timings = {}, {}
print("=== 文字生成（串流）===\n")
with s.post(TARGET + "/v1/chat/completions", json=payload, stream=True,
            timeout=(10, 900)) as r:
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    for line in r.iter_lines(chunk_size=1):
        if not line or not line.startswith(b"data:"):
            continue
        raw = line[5:].strip()
        if raw == b"[DONE]":
            break
        event = json.loads(raw)
        if event.get("error"):
            raise RuntimeError(str(event["error"]))
        for choice in event.get("choices", []):
            piece = (choice.get("delta") or {}).get("content") or ""
            if piece:
                if first_text is None:
                    first_text = time.perf_counter()
                print(piece, end="", flush=True)
        usage = event.get("usage") or usage
        timings = event.get("timings") or timings

elapsed = time.perf_counter() - start
print("\n\n=== 基準測試 ===")
print("輸入 token:", usage.get("prompt_tokens"))
print("輸出 token:", usage.get("completion_tokens"))
print("Prefill:", timings.get("prompt_per_second"), "tok/s")
print("Decode:", timings.get("predicted_per_second"), "tok/s")
print("總時間:", round(elapsed, 2), "秒")
if first_text:
    print("首字延遲:", round(first_text - start, 2), "秒")
if timings.get("draft_n"):
    accept = timings.get("draft_n_accepted", 0)
    print("推測解碼接受率:", round(100 * accept / timings["draft_n"], 1), "%")

## 6. ［選用］比較長文 Prefill 效能

以下用人工重複文字做 **3 種條件的單次測試**，可能與實際文件處理速度不同。輸出上限 64 token，這裡的 Decode 速度不要當成長文生成的數值。

In [ ]:
import uuid, time

paragraph = (
    "Artificial intelligence is transforming scientific research. "
    "Researchers use machine learning to analyze experimental data, "
    "discover new materials, optimize chemical processes, and predict "
    "physical properties. Reliable evaluation requires reproducible "
    "experiments, careful validation, and quantitative comparisons. "
)

for label, chars in [("短文", 100), ("中等", 20000), ("長文", 50000)]:
    doc = (paragraph * (chars // len(paragraph) + 1))[:chars]
    text = f"Test ID: {uuid.uuid4().hex}\n請用繁體中文簡潔摘要以下文章。\n{doc}"
    data = {
        "model": "qwen3.8-flash-next-iq2_xs", "messages": [{"role": "user", "content": text}],
        "max_tokens": 64, "reasoning_effort": "none", "temperature": 0, "stream": False
    }
    t0 = time.perf_counter()
    r = s.post(BASE + "/v1/chat/completions", json=data, timeout=900)
    r.raise_for_status()
    out = r.json()
    tm, usage = out.get("timings", {}), out.get("usage", {})
    print(f"{label}: 輸入 {usage.get('prompt_tokens')} tokens | "
          f"Prefill {tm.get('prompt_per_second')} tok/s | "
          f"Decode {tm.get('predicted_per_second')} tok/s | "
          f"快取重用 {tm.get('cache_n')} | "
          f"總時間 {time.perf_counter() - t0:.2f} 秒")

## 7. ［選用］把模型備份到 Google Drive

請在首次安裝與下載全部完成後再執行。因為**不只模型原始檔，產生的模型資料也會一併複製**，Drive 需要的空間可能大於模型下載大小（約 68GB）。下次執行階段請先還原這些資料，再安裝 Strata 本體。不要直接從 Drive 推論，還原到 Colab 的 SSD 會比較穩定。

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/Strata-backup")
backup.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2",
                "/content/Strata-data/", str(backup) + "/"], check=True)
print("模型資料備份完成:", backup)

### 下次工作階段還原時

重啟 Colab 後，**在執行安裝之前**先掛載 Google Drive 並執行下面這格。先確認 GPU 種類、RAM、可用 SSD 與這次相當。之後取得 Strata 本體，重跑安裝那格。

In [ ]:
# 僅在下次工作階段使用（平常不用執行）
from google.colab import drive
import subprocess
from pathlib import Path

drive.mount("/content/drive")
source = Path("/content/drive/MyDrive/Strata-backup")
assert source.exists(), "Drive 上找不到備份"
Path("/content/Strata-data").mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", str(source) + "/", "/content/Strata-data/"], check=True)
print("還原完成，請繼續安裝 Strata")

---

**用完後**：先執行下面這格關閉 tunnel，再中斷並刪除 Colab 執行階段，停止消耗運算單元。

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
print("tunnel 已關閉")

**用手機開啟**：Google Colab 是網頁服務，不用裝 App。可用 Android／iPhone 瀏覽器開啟並上傳 `.ipynb`（選單不好操作時可切換電腦版網站）。筆記本可以存在手機，但**約 68GB 的模型是下載到 Colab 虛擬機，不是手機**。長時間下載與執行用電腦瀏覽器比較好操作。